# What to Expect in This Notebook

This notebook explains the model-selection reasoning for the Student Scheduler's personalized burnout-prediction system. The available data is small, longitudinal, and specific to one student, so the goal is not to test every machine-learning technique. The goal is to compare a small set of defensible models fairly, select one model for each output, and deploy them.

The three implemented model families are:

- Linear Regression
- Random Forest Regression
- XGBoost Regression

Other techniques are discussed briefly to explain why they are outside the current V1 scope. They do not need to be implemented merely for completeness.

## Design Change: Classification to Two Regression Targets

The project was initially formulated as binary trajectory classification. The original target compared weighted averages of past and future burnout and returned `1` when future burnout was higher and `0` otherwise.

That formulation identified direction, but it discarded magnitude and depended on manually chosen windows and weights. A slight increase and a severe increase produced the same label. It also did not directly match the information the deployed Scheduler should display.

The final V1 design instead contains two regression tasks:

| Output | Target | Meaning |
|---|---|---|
| Tomorrow's burnout | `burnout(t + 1)` | The expected burnout level on the next calendar day |
| Seven-day trajectory | Slope fitted to `burnout(t + 1)` through `burnout(t + 7)` | The expected rate of change in burnout points per day |

The interface can convert the predicted slope into **Increasing**, **Stable**, or **Decreasing**, but that label is presentation logic rather than a third machine-learning model.

These outputs answer different questions. A student may have high predicted burnout but a negative slope, meaning that the student is currently struggling but recovering. Another student may have moderate predicted burnout and a positive slope, meaning that the situation is worsening.

## Goals

The model comparison should answer:

- Which model predicts next-day burnout most accurately on later dates?
- Which model predicts the seven-day burnout slope most accurately?
- Does nonlinear model complexity provide a meaningful improvement over a linear baseline?
- How stable is each model when only a small amount of student history is available?
- Does retaining older semesters through the Heap strategy help, or does the recent-history Stack strategy adapt better?
- Is any performance improvement large enough to justify additional tuning and deployment complexity?

The best model does not have to be the same for both targets. Model selection and evaluation must be performed separately for the next-day level and seven-day slope.

## Data Limitations and Personalization

Even six four-month semesters provide only roughly 720 daily observations before rows are removed for missing targets and chronological holdouts. A first-year student would have substantially less history. Complex models may therefore overfit, and a simple model may remain competitive.

Statistically, the dataset is a **longitudinal repeated-measures sample**, not a collection of independent and identically distributed observations. Ratings recorded on nearby days may exhibit **temporal autocorrelation**, so 720 rows contain less independent information than 720 unrelated observations. This smaller effective sample size increases uncertainty in validation scores and makes high-capacity estimators more vulnerable to variance.

The data may also be **non-stationary**. The joint distribution of workload, wellness, and burnout can change between semesters because courses, routines, and outside circumstances change. This form of distribution shift or concept drift means that a model fitted to older semesters can have low training error while generalizing poorly to the current semester.

V1 focuses on personalized models built from an individual student's ratings and schedule. A generalized cold-start model for new users would require a larger, consented multi-student dataset and remains outside the current scope.

Semester adaptation is not solely a property of the estimator. It also depends on which history is retained, how features are constructed, and when the model is retrained. The Heap-versus-Stack experiment evaluates this history policy; it should not be described as the model automatically adapting by itself.


# Model Candidates

The first experiment uses reasonable baseline configurations. Hyperparameter tuning comes only after the complete training and evaluation pipeline works. All models solving a given target receive the same feature set, chronological periods, and evaluation metrics.

## ✅ Linear Regression

Linear Regression is the baseline for both regression targets. It tests whether the engineered moving averages and workload features already expose enough structure for a simple relationship to perform well.

**Statistical reasoning:** Relative to the tree ensembles, Linear Regression imposes a stronger functional-form assumption and therefore usually has higher bias but lower variance. That tradeoff can be beneficial when the effective sample size is small. If its out-of-sample error is close to the nonlinear models, the simpler estimator is evidence that additional flexibility is not producing a practically useful reduction in bias.

The three-day and seven-day moving averages are likely to be strongly correlated with one another and with current wellness measurements. This **multicollinearity** does not necessarily damage prediction, but it can inflate the sampling variance of individual coefficient estimates and make their signs or magnitudes unstable across training windows. Coefficients should therefore be interpreted as conditional associations rather than causal effects. Variance inflation factors and coefficient stability across chronological folds may be inspected if interpretability becomes important.

**Expectations:** It may perform competitively because the dataset is small and many features summarize recent wellness or workload numerically. If residual plots show systematic curvature or feature-dependent error, that would suggest model bias from an overly restrictive linear specification. If coefficient instability becomes a problem, Ridge Regression can be evaluated later as a separately logged regularized configuration.

**Preprocessing:** Numeric scaling should be packaged with the estimator in an sklearn `Pipeline`. The scaler must be fitted only on training data.

## ✅ Random Forest Regressor

Random Forest represents nonlinear bagged-tree learning. It can model interactions such as high workload becoming more harmful when sleep is falling and stress is already elevated without requiring those interactions to be manually specified.

**Statistical reasoning:** A single deep Decision Tree has low bias but high variance: small changes in the training observations can produce very different splits. Random Forest applies **bootstrap aggregation (bagging)** and feature subsampling so that averaging many partially decorrelated trees reduces this variance. This makes it a reasonable nonlinear alternative when feature interactions exist but a single tree is unstable.

Bagging does not eliminate small-sample risk. Deep trees can still fit noise, and standard out-of-bag estimates rely on random resampling that does not respect the temporal ordering of this dataset. Out-of-bag error should not replace chronological validation here.

**Expectations:** It may improve on Linear Regression if the relationship between wellness, workload, and burnout is nonlinear. The main warning signs are a large training-validation error gap and unstable validation performance across time windows. A standalone Decision Tree may be useful for explanation, but it does not need to become a fourth final candidate.

**Preprocessing:** Scaling is generally unnecessary, but missing-value semantics and feature ordering must remain consistent between training and inference.

## ✅ XGBoost Regressor

XGBoost builds boosted trees sequentially, with later trees correcting errors made by earlier trees. It provides the highest-capacity candidate in the V1 comparison.

**Statistical reasoning:** Boosting sequentially fits residual structure left by earlier weak learners. Compared with the linear baseline, this can reduce approximation bias by learning nonlinearities and interactions. Compared with bagging, however, boosting is more capable of repeatedly fitting difficult observations, including noise and outliers. Its bias reduction can therefore be offset by increased estimator variance when the sample is small.

Shrinkage through the learning rate, shallow trees, row and feature subsampling, regularization, and early stopping control the model's effective complexity. These choices should be treated as regularization decisions, not merely performance knobs.

**Expectations:** It may capture useful nonlinear relationships among recent wellness, exam proximity, task counts, and priority load. It may also overfit badly when observations are limited. Greater complexity is justified only if it produces a meaningful and consistent reduction in out-of-sample error across chronological validation windows rather than one unusually strong split.

**Preprocessing:** Scaling is generally unnecessary. Hyperparameters must be selected using chronological validation rather than the final test period.

# Suboptimal Techniques

## ❌ Neural Networks / MLP

A neural network adds many parameters and training choices without enough personalized observations to justify them. It would create a high overfitting risk and make the final system harder to explain and deploy. Neural approaches can be reconsidered only if the project later obtains substantially more student data.

## ❌ Support Vector Regression

Support Vector Regression could model nonlinear relationships with kernels and might perform reasonably on a small dataset. However, it adds sensitivity to scaling, kernel choice, `C`, `epsilon`, and kernel-specific parameters without answering a distinct V1 research question that the selected linear and tree models do not already cover. It is excluded to keep the comparison focused, not because SVR is incapable of regression or temporal prediction.

## ❌ K-Nearest Neighbors Regression

KNN depends on a meaningful distance measure. With correlated features on different scales and an expanding feature set, nearby rows in feature space may not represent genuinely similar student states. Prediction also becomes more expensive as stored history grows. It is therefore not a priority for V1.

## ❌ Naive Bayes

Standard Naive Bayes algorithms are classifiers, while both final targets are continuous. In addition, wellness and workload features are intentionally related: exams may affect stress, sleep, energy, and time spent simultaneously. Its conditional-independence assumption does not fit the feature design, and converting continuous targets back into categories would discard information.

## Logistic Regression Is No Longer Applicable

Logistic Regression was appropriate for the original binary trajectory target. It is not a regression model for continuous burnout level or slope despite its name. It has therefore been replaced by Linear Regression in the final model comparison. The old classifier may be retained only as a documented legacy experiment, not as a V1 candidate.


# Evaluation and Selection

## Chronological Evaluation

Daily observations are sequential and correlated, so the primary experiment must not randomly shuffle dates. Use earlier observations for training, later observations for validation, and the latest untouched period for final testing.

Random splitting would violate the independence assumption behind ordinary cross-validation because adjacent training and validation rows can contain nearly identical moving-window information. The resulting score would be an **optimistically biased estimate of generalization error**. Chronological evaluation instead estimates performance under the actual deployment condition: predicting observations that occur after the training period.

```text
earliest dates -> training -> later validation -> latest final test
```

Target horizons must also be respected at split boundaries. The next-day target requires training labels to stop before the validation target period. The seven-day slope target requires a seven-day purge so that a training label cannot use burnout observations belonging to the validation period. Apply the same rule between validation and final test data.

If the data volume permits it, expanding-window validation is preferable to choosing a model from a single validation block. The final test period must not be used for feature decisions, model selection, hyperparameter tuning, or threshold selection.

Use identical chronological folds for every candidate so model errors can be compared on the same observations. This creates a **paired comparison** and prevents differences in evaluation periods from being mistaken for differences between estimators. Report the mean and variability of each metric across folds rather than relying only on the best or final fold.

## Metrics

For tomorrow's burnout model:

- MAE is the primary metric because it is directly interpretable in burnout points.
- RMSE is secondary and emphasizes large prediction errors.
- R-squared provides additional context but may be unstable when a small test period has little target variance.

For the seven-day slope model:

- MAE in burnout points per day is the primary metric.
- RMSE in burnout points per day is secondary.
- Increasing/stable/decreasing agreement may be reported as a product-facing secondary metric after applying the configured slope threshold.

Accuracy, precision, recall, F1 score, class imbalance, and stratified splitting belonged to the old classification formulation and are not the primary evaluation tools for these regression targets.

Metric values alone do not describe the error process. Inspect residuals over time, residuals against predicted values, and errors by semester or workload level. Persistent residual autocorrelation suggests that the model has not captured some temporal structure. Systematic underprediction at high burnout levels suggests conditional bias even when overall MAE appears acceptable.

Because the test sets are small and temporally dependent, small metric differences should not automatically be treated as meaningful. Emphasize the magnitude and consistency of improvement. If uncertainty intervals are added later, use a time-aware method such as a block bootstrap rather than independently resampling daily rows.

## Heap Versus Stack

Heap batches retain progressively more historical semesters, while Stack batches retain a more recent window. These overlapping batches are not independent datasets. They test whether older personalized history remains useful or whether forgetting it improves adaptation.

This comparison expresses a statistical tradeoff. Heap increases the sample size and may reduce estimator variance, but stale observations from a shifted distribution can increase bias on the current semester. Stack uses fewer observations and may therefore have higher variance, but focusing on the recent distribution may reduce bias under concept drift. Neither policy is universally better; the preferred policy is the one that produces lower and more stable chronological out-of-sample error.

To control scope, first compare the three candidate algorithms on a common history configuration for each target. After selecting a promising algorithm, compare Heap and Stack using a common validation or test date range where possible. Do not run every model against every overlapping batch merely to create more experiment runs.

## Final Model Selection

The winning model is not automatically the model with the best single score. Selection should consider:

- chronological validation performance;
- stability across time windows;
- performance with limited history;
- overfitting;
- interpretability;
- inference and deployment complexity.

Selection should be based on **generalization error**, not training fit. A complex model should win only when its validation improvement is practically meaningful relative to the scale of the target and sufficiently consistent across folds to outweigh its higher variance and deployment cost. Do not claim statistical significance from a single split.

The final result may select different algorithms for next-day burnout and seven-day slope. A simple model should be preferred when a more complex model does not provide a meaningful and repeatable improvement.
